In [1]:
#@title Gerekli importlarin yapilmasi
!pip install --upgrade pandas==1.3

import warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.core.interactiveshell import InteractiveShell
from statistics import mean

from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual, HTML
import ipywidgets as widgets
from IPython.display import clear_output

InteractiveShell.ast_node_interactivity = "all"
warnings.filterwarnings("ignore")

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
#@title Baslangic Değerlerini Ata isimli fonksiyon
def Baslangic_Degerlerini_Ata(baslangic_bakiyesi = 100.0 , basilacak_mi =True):
    global hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    hiperparametreler = {
        "baslangic_bakiyesi": baslangic_bakiyesi,
        "kar_hedefi": 2,
        "fuzyona_girecek_yuzde": 1 ,
        "fuzyona_girmesi_icin_dusecek_yuzde": 1,
        "fuzyon_geri_alim_icin_dusulecek_yuzde":1,
        "parca_sayisi": 5,
        "ilk_satis_sicrama_yuzdesi": 1,
        "satis_parcalar_arasi_yuzde_orani":0.5,
        "satilan_parcanin_tekrar_geri_alinacagi_yuzde": 0.5,
        "komisyon_katsayisi": 0.999
    }

    fuzyon = {
        "statu": "",
        "fiyat": 0.0
    }

    kunye = {'son_aksiyon': "sistem baslamadi",
            'son_okunan_fiyat': 0.0,
             'hedef': 0.0,
             'yukselis_aksiyon_fiyati': 0.0,
             'yukselisteki_ilk_aksiyon': '',
             'inis_aksiyon_fiyati': 0.0,
             'inisteki_ilk_aksiyon': '',
             'p1_miktari': 0.0,
             'p2_miktari': baslangic_bakiyesi,
             'nakde_cevrilebilir_deger': 0.0
             }

    satis_plani = {
        "adet": 0.0,
        "fiyat": 0.0,
        "satis_sonrasi_ele_gececek_p2" : 0.0
    }

    alis_plani = {
        "adet": 0.0,
        "fiyat": 0.0
    }

    hiperparametreler = pd.Series(hiperparametreler)
    
    kunye = pd.Series(kunye)
    fuzyon = pd.Series(fuzyon)[0:0]
    alis_plani = pd.DataFrame(data=[alis_plani.values()], columns=list(alis_plani.keys()))[0:0]
    satis_plani = pd.DataFrame(data=[satis_plani.values()], columns=list(satis_plani.keys()))[0:0]

    # Başlangıçta gelenlere de aynı değerleri atayalım ki tipleri aynı olsun.
    gelen_kunye = kunye.copy(deep = True)
    gelen_fuzyon = fuzyon.copy(deep= True)[0:0]
    gelen_alis_plani = alis_plani[0:0].copy(deep= True)
    gelen_satis_plani = satis_plani[0:0].copy(deep= True)

    if basilacak_mi:
        print("Değişkenlere başlangıç değerleri atandı.")

In [3]:
#@title Satış Planı Oluşturucu Fonksiyon
def Satis_Plani_Olusturucu(p1 , gelen_fiyat , hedef):
    spamyo = hiperparametreler["satis_parcalar_arasi_yuzde_orani"] # Satış parçaları arası min yüzde oranı
    komisyon_orani = hiperparametreler["komisyon_katsayisi"]
    ilk_satis_sicrama_yuzdesi = hiperparametreler["ilk_satis_sicrama_yuzdesi"]

    ilk_parcanin_satis_fiyati = gelen_fiyat*(100+ilk_satis_sicrama_yuzdesi)/100

    satis_fiyatlari = [ilk_parcanin_satis_fiyati]
    while mean(satis_fiyatlari)*komisyon_orani*p1 < hedef and max(satis_fiyatlari)< gelen_fiyat*1.04:
        listeye_eklenecek_eleman = max(satis_fiyatlari)*(100+spamyo)/100
        satis_fiyatlari.append(listeye_eklenecek_eleman)

    satilacak_birim_parca_miktari = p1/len(satis_fiyatlari)

    satis_plani = pd.DataFrame()
    satis_plani["fiyat"] = satis_fiyatlari
    satis_plani["adet"] = satilacak_birim_parca_miktari
    satis_plani["satis_sonrasi_ele_gececek_p2"] = satis_plani["fiyat"]*satis_plani["adet"]*komisyon_orani

    yeni_hedef = satis_plani["satis_sonrasi_ele_gececek_p2"].sum()

    return yeni_hedef ,satis_plani

In [4]:
#@title Duzluge Cikarici isimli fonksiyon
def Duzluge_Cikarici():
    global gelen_fiyat,  hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    komisyon_orani = hiperparametreler["komisyon_katsayisi"]


    # Eldeki baslangic bakiyesinin tamamı ile p1 satin alalım.
    p1 = hiperparametreler["baslangic_bakiyesi"]/gelen_fiyat
    p1 = p1*komisyon_orani


    hedef_deger = hiperparametreler["baslangic_bakiyesi"]*(hiperparametreler["kar_hedefi"] +100)/100
    hedef ,yeni_satis_plani = Satis_Plani_Olusturucu(p1 , gelen_fiyat ,hedef_deger)


    # Hadi simdi de fuzyon planimizi yapalim.
    fuzyona_girecegi_fiyat = gelen_fiyat*(100 - hiperparametreler["fuzyona_girmesi_icin_dusecek_yuzde"])/100
    fuzyona_girecegi_fiyat = round(fuzyona_girecegi_fiyat,2)
    yeni_fuzyon = {
        "statu": "fuzyona_girmedi",
        "fiyat": fuzyona_girecegi_fiyat
    }

    # Satis planimiz hazir , fuzyon planimiz hazir , alis planimiz bos cunku satis yapilip alinmayi bekleyen parca yok. Kunyemizi de tamamlayarak tum ciktilari halledelim.
    yeni_kunye = pd.Series()
    yeni_kunye["son_aksiyon"] = "düzlüğe çıkarıldı"
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = hedef_deger
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = fuzyona_girecegi_fiyat
    yeni_kunye["inisteki_ilk_aksiyon"] = "fuzyona_girecek"
    yeni_kunye["p1_miktari"] = p1
    yeni_kunye["p2_miktari"] = 0
    yeni_kunye["nakde_cevrilebilir_deger"] = p1*gelen_fiyat*komisyon_orani

    # Haydi sistemin grafigini cizelim.
    # X ekseninde küsüratlı değerler çıktığı için ve bizde sadece noktalara karsi gelen degerleri gormek istedigimiz icin bu sekilde bir eksen plani yapiyoruz.

    grafik_verisi = pd.DataFrame()
    grafik_verisi["fiyat"] = satis_plani["fiyat"]
    grafik_verisi["aksiyon"] = "satilacak"

    grafik_verisi = grafik_verisi.append({'fiyat': gelen_fiyat, 'aksiyon': "mevcut fiyat"}, ignore_index=True)
    grafik_verisi = grafik_verisi.append({'fiyat': fuzyona_girecegi_fiyat, 'aksiyon': "fuzyona girilecek fiyat"}, ignore_index=True)

    grafik_verisi["y_ekseni"] = 1
    grafik_verisi = grafik_verisi.round(4)

    fig = px.scatter(grafik_verisi, x="fiyat", y="y_ekseni", color="aksiyon", symbol="aksiyon",  height=300, template="plotly_white", title="Satis Plani")

    fig.update_layout(xaxis=dict(
        tickmode='array',
        title="",
        tickvals=grafik_verisi["fiyat"],
        ticktext=grafik_verisi["fiyat"]
    ),
        yaxis=dict(
        visible=False
    ),
        legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title=""
    ),
    )

    grafik = fig.to_dict()

    gelen_alis_plani = alis_plani[0:0]
    gelen_kunye = yeni_kunye.copy(deep= True)
    gelen_satis_plani = yeni_satis_plani.copy(deep = True)
    gelen_fuzyon = pd.Series(yeni_fuzyon)

In [5]:
#@title Fuzyona Giris isimli fonksiyon
def Fuzyona_Giris():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    p1_miktari = kunye["p1_miktari"]
    fuzyona_girecek_yuzde = hiperparametreler["fuzyona_girecek_yuzde"]
    hedef = kunye["hedef"]
    komisyon_orani = hiperparametreler["komisyon_katsayisi"]

    # Fuzyona girecek p1 miktarini bulalım.
    fuzyona_girecek_p1_miktari = p1_miktari*fuzyona_girecek_yuzde/100

    # Haydi o zaman gelen fiyattan fuzyona girelim.
    fuzyon_sonrasi_ele_gecen_p2 = fuzyona_girecek_p1_miktari*gelen_fiyat*komisyon_orani

    # Peki elde ne kadar p1 kaldı? Bunu bulalım sonra hedefe ulaşmak için bu p1'i ortalama kaçtan satmak lazım onu bulalım.
    elde_kalan_p1 = p1_miktari - fuzyona_girecek_p1_miktari
    goreceli_hedef = hedef - fuzyon_sonrasi_ele_gecen_p2
    hedef ,yeni_satis_plani = Satis_Plani_Olusturucu(elde_kalan_p1 , gelen_fiyat ,goreceli_hedef)

    # Satis planini olusturduk. Cok guzel. Simdi de fuzyon planimizi olusturalim.
    yeni_fuzyon = fuzyon.copy(deep=True)
    yeni_fuzyon["statu"] = "fuzyonda"
    yeni_fuzyon["fiyat"] = round(gelen_fiyat * (100 - hiperparametreler["fuzyon_geri_alim_icin_dusulecek_yuzde"])/100 ,2)

    # Süper! Haydi simdi kunyemizi de olusturalim.
    yeni_kunye = kunye.copy(deep=True)
    
    yeni_kunye["son_aksiyon"] = "füzyona girildi"
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = kunye["hedef"]
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = yeni_fuzyon["fiyat"]
    yeni_kunye["inisteki_ilk_aksiyon"] = "eve_donecek"
    yeni_kunye["p1_miktari"] = elde_kalan_p1
    yeni_kunye["p2_miktari"] = fuzyon_sonrasi_ele_gecen_p2
    yeni_kunye["nakde_cevrilebilir_deger"] = elde_kalan_p1*gelen_fiyat*komisyon_orani + fuzyon_sonrasi_ele_gecen_p2

    gelen_alis_plani = gelen_alis_plani[0:0]
    gelen_satis_plani = yeni_satis_plani.copy(deep= True)
    gelen_fuzyon = yeni_fuzyon.copy(deep = True)
    gelen_kunye = yeni_kunye.copy(deep = True)

In [6]:
#@title Eve Donus isimli fonksiyon
def Eve_Donus():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani
    
    eldeki_p2 = kunye["p2_miktari"]
    eve_donus_sonrasi_ele_gecen_p1 = (eldeki_p2/gelen_fiyat)*hiperparametreler["komisyon_katsayisi"]

    # Peki bu işlem sonrası elimizde toplam ne kadar p1 oldu?
    toplam_p1 = eve_donus_sonrasi_ele_gecen_p1 + kunye["p1_miktari"]

    # Şimdi bu eldeki p1'leri hedefe ulaşmak için nasıl bir satış planı oluşturmalıyız ona bakalım.
    hedef = kunye["hedef"]
    komisyon_orani = hiperparametreler["komisyon_katsayisi"]
    hedef , yeni_satis_plani = Satis_Plani_Olusturucu(toplam_p1 , gelen_fiyat ,hedef)


    # Satis planini olusturduk. Cok guzel. Simdi de fuzyon planimizi olusturalim.
    yeni_fuzyon = fuzyon.copy(deep=True)
    yeni_fuzyon["statu"] = "fuzyona_girmedi"
    yeni_fuzyon["fiyat"] = round(gelen_fiyat * (100 - hiperparametreler["fuzyona_girmesi_icin_dusecek_yuzde"])/100 ,2)
  
    yeni_kunye = pd.Series()
    yeni_kunye["son_aksiyon"] = "eve dönüldü"
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = kunye["hedef"]
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = yeni_fuzyon["fiyat"]
    yeni_kunye["inisteki_ilk_aksiyon"] = "fuzyona_girecek"
    yeni_kunye["p1_miktari"] = toplam_p1
    yeni_kunye["p2_miktari"] = 0
    yeni_kunye["nakde_cevrilebilir_deger"] = toplam_p1*gelen_fiyat*komisyon_orani

    gelen_satis_plani = yeni_satis_plani.copy(deep= True)
    gelen_fuzyon = yeni_fuzyon.copy(deep = True)
    gelen_kunye = yeni_kunye.copy(deep = True)

In [7]:
#@title Satis Yapilacak isimli fonksiyon
def Satis_Yapilacak():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    # Haydi şimdi mevcut fiyata göre hangileri satılmalı onlara bakalım.
    satilacaklar = satis_plani[gelen_fiyat >= satis_plani["fiyat"]][["adet", "fiyat"]]

    # Bu fiyattan satis yaptigimizda elimize gececek tutari da bulalim.
    satilacaklar["bu_satistan_sonra_ele_gececek_tutar"] = satilacaklar["adet"]*gelen_fiyat*hiperparametreler["komisyon_katsayisi"]

    # Satış planımızdan satılacak (aslında satıldı) olanları çıkartalım. Bunu yaparken fiyatları eşleşenleri sildirelim.
    satilacaklar_fiyatlar = satilacaklar["fiyat"].tolist()
    silinecek_satirlar = satis_plani[satis_plani["fiyat"].isin(satilacaklar_fiyatlar)]
    yeni_satis_plani = satis_plani.drop(silinecek_satirlar.index).reset_index(drop=True)
    # Satış planımızdan satılacak (aslında satıldı) olanları çıkartalım. Bunu yaparken fiyatları eşleşenleri sildirelim.

    #Asagidaki kod "bu_satistan_sonra_ele_gececek_tutar" isimli degiskeni "adet" ismiyle degistirir. Bu kısım cok aklimi karistirmisti.
    #Yani asagidaki islemin sonunda adet-fiyat isimli iki kolonlu bir yapi olusur.
    alis_planina_eklenecek_veri = satilacaklar[["bu_satistan_sonra_ele_gececek_tutar", "fiyat"]].rename(columns={"bu_satistan_sonra_ele_gececek_tutar": "adet"})

    # satilan_parcanin_tekrar_geri_alinacagi_yuzde parametresini de ekleyelim. Sonuçta sattığımız fiyattan geri alırsak bu tam amelelik olur.
    alis_planina_eklenecek_veri["fiyat"] = alis_planina_eklenecek_veri["fiyat"]*(100 -hiperparametreler["satilan_parcanin_tekrar_geri_alinacagi_yuzde"])/100

    # Tüm elemanlar 0.0 yada "0.0" ise df'yi temizlesin. Bunu yazma amacımız ilk sistem daha sıfırken alış planını yaratırken 0.0 değer atıyoruz. Alış planı için
    if alis_plani.all().all() == 0.0 or alis_plani.all().all() == "0.0":
        alis_plani = alis_plani[0:0]

    yeni_alis_plani = pd.concat([alis_plani, alis_planina_eklenecek_veri], axis=0).reset_index(drop=True)

    komisyon_orani = hiperparametreler["komisyon_katsayisi"]

    yeni_kunye = kunye.copy(deep=True)
    yeni_kunye["son_aksiyon"] = "satış yapıldı"
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = kunye["hedef"]
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = yeni_alis_plani["fiyat"].max()
    yeni_kunye["inisteki_ilk_aksiyon"] = "duz_alis"
    yeni_kunye["p1_miktari"] = yeni_kunye["p1_miktari"] - satilacaklar["adet"].sum()
    yeni_kunye["p2_miktari"] = yeni_kunye["p2_miktari"] + satilacaklar["bu_satistan_sonra_ele_gececek_tutar"].sum()
    yeni_kunye["nakde_cevrilebilir_deger"] = yeni_kunye["p1_miktari"]*gelen_fiyat*komisyon_orani + yeni_kunye["p2_miktari"]

    gelen_satis_plani = yeni_satis_plani.copy(deep=True)
    gelen_alis_plani = yeni_alis_plani.copy(deep=True)
    gelen_kunye = yeni_kunye.copy(deep=True)

In [8]:
#@title Nakde Cevrilebilir Deger isimli fonksiyon
def NCD_Firlamasi():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    if kunye["p2_miktari"] > 0:
        ele_gecen_p1 = hiperparametreler["komisyon_katsayisi"] * kunye["p2_miktari"]/gelen_fiyat
        kunye["p2_miktari"] = 0
        kunye["p1_miktari"] = kunye["p1_miktari"] + ele_gecen_p1

    # Buradan sonrasi duzluge cikarticidan kopyalanan kisim gibi
    p1 = kunye["p1_miktari"]
    komisyon_orani = hiperparametreler["komisyon_katsayisi"]

    anlik_nakde_cevrilebilir_deger = kunye["p1_miktari"]*gelen_fiyat*hiperparametreler["komisyon_katsayisi"] + kunye["p2_miktari"]
    # Zaten elimizdeki nçd hedeften daha yüksek olduğu için ve hedefe ulaştığımız için o yüzden yeni hedefimizi eski hedef üzerinden değil de nçd üzerinden çarparak veririz.
    hedef_deger = anlik_nakde_cevrilebilir_deger*(hiperparametreler["kar_hedefi"] +100)/100
    komisyon_orani = hiperparametreler["komisyon_katsayisi"]
    hedef , yeni_satis_plani = Satis_Plani_Olusturucu(p1 , gelen_fiyat ,hedef_deger)
    
    # Hadi simdi de fuzyon planimizi yapalim.
    fuzyona_girecegi_fiyat = gelen_fiyat*(100 - hiperparametreler["fuzyona_girmesi_icin_dusecek_yuzde"])/100
    fuzyona_girecegi_fiyat = round(fuzyona_girecegi_fiyat,2)
    
    yeni_fuzyon = {
        "statu": "fuzyona_girmedi",
        "fiyat": fuzyona_girecegi_fiyat
    }

    yeni_fuzyon = pd.Series(yeni_fuzyon)


    # Satis planimiz hazir , fuzyon planimiz hazir , alis planimiz bos cunku satis yapilip alinmayi bekleyen parca yok. Kunyemizi de tamamlayarak tum ciktilari halledelim.

    # Simdi kunyemize veri eklemeye baslayalim.
    yeni_kunye = pd.Series()
    yeni_kunye["son_aksiyon"] = "hedefe ulaşıldı"
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = hedef_deger
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = fuzyona_girecegi_fiyat
    yeni_kunye["inisteki_ilk_aksiyon"] = "fuzyona_girecek"
    yeni_kunye["p1_miktari"] = p1
    yeni_kunye["p2_miktari"] = 0
    yeni_kunye["nakde_cevrilebilir_deger"] = p1*gelen_fiyat*komisyon_orani

    gelen_alis_plani = alis_plani[0:0]
    gelen_kunye = yeni_kunye.copy(deep=True)
    gelen_satis_plani = yeni_satis_plani.copy(deep=True)
    gelen_fuzyon = yeni_fuzyon.copy(deep=True)

In [9]:
#@title Alış yapma işleminin yapılacağı fonksiyon
def Alis_Yap():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani

    komisyon = hiperparametreler["komisyon_katsayisi"]
    alis_plani_klon = alis_plani.copy(deep = True)

    alis_plani_klon["alis_yapilacak_mi"] = alis_plani_klon["fiyat"] >= gelen_fiyat
    alinanlar = alis_plani_klon[alis_plani_klon["alis_yapilacak_mi"] == True]

    alinanlar["ele_gececek_p1"] = komisyon*alis_plani_klon["adet"]/gelen_fiyat

    # Eleman sayılarını belirleyelim.
    alis_listesi_eleman_sayisi = len(alis_plani_klon.index)
    alinacak_eleman_sayisi = len(alinanlar.index)
    # Eleman sayılarını belirleyelim.

    ele_gecen_p1 = alinanlar["ele_gececek_p1"].sum()
    elden_giden_p2 = alinanlar["adet"].sum()

    #Bu satış sonrası elimizde toplam kaç p1 , p2 oldu , onu hesaplayalım.
    toplam_p1 = kunye["p1_miktari"] + ele_gecen_p1
    toplam_p2 = kunye["p2_miktari"] - elden_giden_p2
    goreceli_hedef = kunye["hedef"] - toplam_p2

    hedef , yeni_satis_plani = Satis_Plani_Olusturucu(toplam_p1 , gelen_fiyat ,goreceli_hedef)

    
    #Yeni alış planımızı da görelim
    yeni_alis_plani =  alis_plani_klon[alis_plani_klon["alis_yapilacak_mi"] == False][["adet","fiyat"]]

    #Künyemizi de yapalım.
    yeni_kunye = pd.Series()
    yeni_kunye["son_aksiyon"] = "kısmi alış yapıldı" # bu kısım altta netlestirilecek
    yeni_kunye["son_okunan_fiyat"] = gelen_fiyat
    yeni_kunye["hedef"] = kunye["hedef"]
    yeni_kunye["yukselis_aksiyon_fiyati"] = yeni_satis_plani["fiyat"].min()
    yeni_kunye["yukselisteki_ilk_aksiyon"] = "duz_satis"
    yeni_kunye["inis_aksiyon_fiyati"] = yeni_alis_plani["fiyat"].max() # bu kısım altta netlestirilecek
    yeni_kunye["inisteki_ilk_aksiyon"] = "duz_alis" # bu kısım altta netlestirilecek
    yeni_kunye["p1_miktari"] = toplam_p1
    yeni_kunye["p2_miktari"] = toplam_p2
    yeni_kunye["nakde_cevrilebilir_deger"] = toplam_p1*gelen_fiyat*komisyon + toplam_p2

    yeni_fuzyon = fuzyon.copy(deep = True)
    
    # Eğer tüm elemanlar alınmışsa künye ve füzyon planı da ona göre değişecek.
    if (alis_listesi_eleman_sayisi == alinacak_eleman_sayisi) and alinacak_eleman_sayisi > 0:
        yeni_kunye["son_aksiyon"] = "tüm elemanlar alındı"
        yeni_alis_plani = alis_plani[0:0]

        # Eğer füzyona girmemişse yeniden füzyona gireceği fiyatı belirleriz.
        if fuzyon["statu"] == "fuzyona_girmedi":
            fuzyona_girecegi_fiyat = gelen_fiyat*(100 - hiperparametreler["fuzyona_girmesi_icin_dusecek_yuzde"])/100
            fuzyona_girecegi_fiyat = round(fuzyona_girecegi_fiyat,2)
            yeni_fuzyon = {
                "statu": "fuzyona_girmedi",
                "fiyat": fuzyona_girecegi_fiyat
            }
            yeni_fuzyon = pd.Series(yeni_fuzyon)

            yeni_kunye["inisteki_ilk_aksiyon"] = "fuzyona_girecek"
            yeni_kunye["inis_aksiyon_fiyati"] = fuzyona_girecegi_fiyat
            
        else:
            # Eğer füzyona girmişse ellemeyiz.
            yeni_kunye["inisteki_ilk_aksiyon"] = "eve_donecek"
            yeni_kunye["inis_aksiyon_fiyati"] = fuzyon["fiyat"]
            
  
    gelen_kunye = yeni_kunye.copy(deep=True)
    gelen_fuzyon = yeni_fuzyon.copy(deep = True)
    gelen_satis_plani = yeni_satis_plani.copy(deep=True)
    gelen_alis_plani = yeni_alis_plani.copy(deep=True)

In [10]:
#@title Pas Geç isimli fonksiyon
def Pas_Gec():
    global gelen_fiyat, hiperparametreler , fuzyon , kunye , satis_plani , alis_plani, gelen_fuzyon , gelen_kunye ,gelen_satis_plani, gelen_alis_plani
    
    gelen_kunye = kunye.copy(deep = True)
    
    gelen_kunye["son_aksiyon"] = "pas geçildi"
    gelen_kunye["nakde_cevrilebilir_deger"] = gelen_kunye["p1_miktari"]*gelen_fiyat*hiperparametreler["komisyon_katsayisi"] + gelen_kunye["p2_miktari"]

    gelen_fuzyon = fuzyon.copy(deep= True)
    gelen_alis_plani = alis_plani.copy(deep= True)
    gelen_satis_plani = satis_plani.copy(deep= True)

In [11]:
#@title Stilleri Ayarlayalım
def Df_Stilcisi(df , yuvarlanacak_sayi = 6):

    if isinstance(df, pd.Series):
        df = pd.DataFrame(df , columns = ["Değerler"])

    df = df.round(yuvarlanacak_sayi)
    baslik_ozellikleri = [('font-size', '12px'),('background-color', '#579BB1'),('color','#ECE8DD'),("border", "1px solid black")]
    hucre_ozellikleri = [('font-size', '12px')]
    index_ozellikleri = [('text-align', 'left'),('font-size', '12px'),('color','#ECE8DD')]

    dfstyle = [dict(selector="thead", props=baslik_ozellikleri), dict(selector="td", props=hucre_ozellikleri),dict(selector="tbody", props= [("border", "1px solid black")]) ,dict(selector="tbody th", props=index_ozellikleri),]
    df = df.style.set_table_styles(dfstyle)

    #display(fuzyon_kiyaslamasi.style.set_properties(**{'background-color': '#00ADB5', 'color': '#EEEEEE'})) renk için
    return df

In [12]:
#@title Öncesi Sonrası isimli fonksiyon
def Oncesi_Sonrasi():
    global hiperparametreler , kunye , fuzyon, satis_plani , alis_plani , gelen_fiyat , gelen_kunye ,gelen_fuzyon , gelen_satis_plani , gelen_alis_plani


    if kunye.to_frame().equals(gelen_kunye.to_frame()):
        print("Künye planında bir değişiklik yok.")
        display(Df_Stilcisi(kunye))
    else:
        
        print("Künye Kiyaslamasi")
        kunye_kiyaslamasi =  pd.concat([kunye.to_frame() , gelen_kunye.to_frame()] , axis =1)
        kunye_kiyaslamasi.columns  = ["Eski Kunye", "Yeni Kunye"]
        display(Df_Stilcisi(kunye_kiyaslamasi))

    if fuzyon.to_frame().equals(gelen_fuzyon.to_frame()):
        
        if len(fuzyon.to_frame()) == 0:
            print("Şu an için bir Füzyon planı yok.")
        else:
            print("Füzyon planında bir değişiklik yok.")
            display(Df_Stilcisi(fuzyon))

    else:
        print("Fuzyon Kiyaslamasi")
        fuzyon_kiyaslamasi =  pd.concat([fuzyon.to_frame() ,gelen_fuzyon.to_frame()] , axis =1)
        fuzyon_kiyaslamasi.columns  = ["Eski Fuzyon Plani", "Yeni Fuzyon Plani"]
        display(Df_Stilcisi(fuzyon_kiyaslamasi))

    if satis_plani.equals(gelen_satis_plani):
        if len(satis_plani) == 0:
            print("Şu an için bir satış planı yok.")
        else:
            print("Satış planında bir değişiklik yok.")
            display(Df_Stilcisi(satis_plani))
    else:
        print("Satış Planı Kiyaslamasi")
        satis_plani_kiyaslamasi =  pd.concat([satis_plani ,gelen_satis_plani] ,keys=['Eski', 'Yeni'], axis = 0)
        display(Df_Stilcisi(satis_plani_kiyaslamasi))


    if alis_plani.equals(gelen_alis_plani):
        if len(alis_plani) == 0:
            print("Şu an için bir alış planı yok.")
        else:
            print("Alış planında bir değişiklik yok.")
            display(Df_Stilcisi(alis_plani))
    else:
        print("Alış Planı Kiyaslamasi")
        alis_plani_kiyaslamasi =  pd.concat([alis_plani ,gelen_alis_plani] ,keys=['Eski', 'Yeni'])
        display(Df_Stilcisi(alis_plani_kiyaslamasi))

In [13]:
#@title Mevcut Durum
def Mevcut_Durum():
    global hiperparametreler , kunye , fuzyon, satis_plani , alis_plani , gelen_fiyat , gelen_kunye ,gelen_fuzyon , gelen_satis_plani , gelen_alis_plani

    print("Değerler Kaydedildi.")

    print("Künye")
    display(Df_Stilcisi(kunye))
    if len(fuzyon.to_frame()) == 0:
        print("Şu an için bir Füzyon planı yok.")
    else:
        print("Fuzyon")
        display(Df_Stilcisi(fuzyon))

    if len(satis_plani) == 0:
        print("Şu an için bir satış planı yok.")
    else:
        print("Satış Planı")
        display(Df_Stilcisi(satis_plani))


    if len(alis_plani) == 0:
        print("Şu an için bir alış planı yok.")
    else:
        print("Alış Planı")
        display(Df_Stilcisi(alis_plani))

In [14]:
#@title Gelen Degerleri Kaydet
def Gelen_Degerleri_Kaydet(basilacak_mi = True):
    global hiperparametreler , kunye , fuzyon, satis_plani , alis_plani , gelen_fiyat , gelen_kunye ,gelen_fuzyon , gelen_satis_plani , gelen_alis_plani
    kunye = gelen_kunye.copy(deep = True)
    alis_plani = gelen_alis_plani.copy(deep = True)
    satis_plani = gelen_satis_plani.copy(deep = True)
    fuzyon = gelen_fuzyon.copy(deep = True)
    if basilacak_mi:
        Mevcut_Durum()

In [15]:
#@title Yöneylem Araştırması Fonksiyonu
def Operation_Research(fiyat , basilacak_mi = True):
    global hiperparametreler , kunye , fuzyon, satis_plani , alis_plani , gelen_fiyat , gelen_kunye ,gelen_fuzyon , gelen_satis_plani , gelen_alis_plani
    gelen_fiyat = fiyat

    anlik_nakde_cevrilebilir_deger = kunye["p1_miktari"]*gelen_fiyat*hiperparametreler["komisyon_katsayisi"] + kunye["p2_miktari"]

    if kunye["hedef"] == 0.0:
        if basilacak_mi:
            print("Sistem sifirdan basliyor.\nDüzlüğe çıkar fonksiyonu çalışacak.")
        Duzluge_Cikarici()
        #fig = go.Figure(grafik)
        #fig.show()

    elif kunye['yukselis_aksiyon_fiyati'] > gelen_fiyat and gelen_fiyat > kunye['inis_aksiyon_fiyati']:
        if basilacak_mi:
            print(f"Gelen fiyat ({gelen_fiyat}) minimum ve maximum araliginda oldugundan dolayi pas geçilecek.\nmin = {kunye['inis_aksiyon_fiyati']} , max = {kunye['yukselis_aksiyon_fiyati']}")
        # Başlangıçta gelenlere de aynı değerleri atayalım print edince eski değerler kalmasın.
        Pas_Gec()

    elif gelen_fiyat <= kunye["inis_aksiyon_fiyati"] and kunye["inisteki_ilk_aksiyon"] == "duz_alis":
        if basilacak_mi:
            print("Düz alış yapacağız.")
        Alis_Yap()
      
    elif (fuzyon["statu"] == "fuzyona_girmedi" and gelen_fiyat <= fuzyon["fiyat"]) and kunye["inisteki_ilk_aksiyon"] != "duz_alis":
        if basilacak_mi:
            print("Fuzyona Giriyoruz")
        Fuzyona_Giris()
    
    elif fuzyon["statu"] == "fuzyonda" and gelen_fiyat <= fuzyon["fiyat"]:
        if basilacak_mi:
            print("Eve dönüyoruz.")
        Eve_Donus()
     
    elif kunye["hedef"] > 0 and anlik_nakde_cevrilebilir_deger >= kunye["hedef"]:
        if basilacak_mi:
            print(f"Nakde çevrilebilir değer ({anlik_nakde_cevrilebilir_deger}) hedef değerimizden ({kunye['hedef']}) daha yüksek. NCD firmalası fonksiyonu çalışacak.")
        NCD_Firlamasi()
    
    elif gelen_fiyat>= kunye['yukselis_aksiyon_fiyati'] and kunye["yukselisteki_ilk_aksiyon"] == "duz_satis":
        if basilacak_mi:
            print("Düz satış yapacağız.")
        Satis_Yapilacak()
    else:
        if basilacak_mi:
            print("Henuz bir planimiz yok :)")
        NCD_Firlamasi() ## silinecek

    if basilacak_mi:
        Oncesi_Sonrasi()

In [ ]:
#@title Arayüzümüz
# Tarama alanı
display(HTML("<style>.mavi_buton { color:#F5F5F5 ; background-color:#EB455F }</style>"))
display(HTML("<style>.kirmizi_buton { color:#F5F5F5 ; background-color:#2B3467 }</style>"))
display(HTML("<style>.yukseklik { height: 2000px ; width: 1200px; overflow: visible}</style>"))

display(HTML("<style>.lm-TabBar-tabLabel { color:#F5F5F5}</style>"))
                     
fiyat_girisi = widgets.BoundedFloatText(
            value=100,
            min=0,
            max=1000.0,
            step=0.5,
            description='Fiyatı Giriniz:',
            disabled=False
        )

tarama_butonu = widgets.Button(description="Tara").add_class("kirmizi_buton")
output = widgets.Output()
output2 = widgets.Output()

def taramaya_baslayalim(b):
    with output:
        output.clear_output()
        Operation_Research(fiyat_girisi.value)
    
tarama_butonu.on_click(taramaya_baslayalim)
# Tarama alanı

# Baslangic Degerleri alanı
baslangic_degerleri = widgets.Button(description="Sıfırla").add_class("kirmizi_buton")

def baslangic_degerleri_ata(b):
    with output:
        output.clear_output()
        Baslangic_Degerlerini_Ata()
    
    with output2:
        output2.clear_output()
        display(Df_Stilcisi(hiperparametreler,2))

baslangic_degerleri.on_click(baslangic_degerleri_ata)
# Baslangic Degerleri alanı


# Kaydet butonunun alanı
kaydet_butonu = widgets.Button(description="Kaydet").add_class("kirmizi_buton")

def kaydet(b):
    with output:
        output.clear_output()
        Gelen_Degerleri_Kaydet()

kaydet_butonu.on_click(kaydet)
# Kaydet butonunun alanı


# Ust Aksiyon Fiyatina Atla
ust_aksiyon_fiyati = widgets.Button(description="Ust Aksiyona Atla").add_class("mavi_buton")

def ust_aksiyon(b):
    with output:
        output.clear_output()
        fiyat_girisi.value = kunye["yukselis_aksiyon_fiyati"]
        Operation_Research(kunye["yukselis_aksiyon_fiyati"])
    
ust_aksiyon_fiyati.on_click(ust_aksiyon)
# Ust Aksiyon Fiyatina Atla


# Alt Aksiyon Fiyatina Atla
alt_aksiyon_fiyati = widgets.Button(description="Alt Aksiyona Atla").add_class("mavi_buton")

def alt_aksiyon(b):
    with output:
        output.clear_output()
        fiyat_girisi.value = kunye["inis_aksiyon_fiyati"]
        Operation_Research(kunye["inis_aksiyon_fiyati"])


alt_aksiyon_fiyati.on_click(alt_aksiyon)
# Alt Aksiyon Fiyatina Atla

aksiyon_butonlari_bilgi = widgets.Label("Alt veya üst aksiyon fiyatına atlamak için aşağıdaki butonlari kullanabilirsiniz.")
yatay_butonlar_bilgi = widgets.Label("Ana işlemler için alttaki butonları kullanabilirsiniz.")

yatay_butonlar = widgets.HBox([tarama_butonu,baslangic_degerleri,kaydet_butonu])
alt_ust_aksiyon_butonlari = widgets.HBox([alt_aksiyon_fiyati,ust_aksiyon_fiyati])

tab = widgets.Tab().add_class("yukseklik")
tab.children = [output , output2]
tab.set_title(0,"Kıyaslama")
tab.set_title(1,"Hiperparametreler")

widgets.VBox([fiyat_girisi , aksiyon_butonlari_bilgi ,alt_ust_aksiyon_butonlari,yatay_butonlar_bilgi,yatay_butonlar,tab])

In [ ]:
from google.colab import drive
import sqlite3
drive.mount('/content/drive')
conn = sqlite3.connect("/content/drive/MyDrive/veritabani.db")

In [ ]:
fiyat_verisi = pd.read_pickle("/content/drive/MyDrive/BTCUSDT.pkl")
fiyat_verisi = fiyat_verisi.iloc[:,[0,1,2,3,4]]
fiyat_verisi.columns = ["Tarih","Acilis_Zamani","En_Yuksek_Fiyat","En_Dusuk_Fiyat","Kapanis_Fiyati"]
fiyat_verisi = fiyat_verisi.sort_values(by=['Tarih'])
#fiyat_verisi = fiyat_verisi[fiyat_verisi["Tarih"] < "2021-01-01 03:00:00"]

In [ ]:
sorgu_metni = '''
SELECT Tarih,
       Acilis_Fiyati,
       En_Yuksek_Fiyat,
       En_Dusuk_Fiyat,
       Kapanis_Fiyati
  FROM Data
 WHERE Para_Cifti = 'AVAXUSDT'
 Order By Tarih
'''

#fiyat_verisi = pd.read_sql_query(sorgu_metni, conn)

# Veriyi istediğimiz şekle getirelim.
veri =fiyat_verisi.set_index('Tarih')
veri = veri.stack().reset_index()
veri.columns = ["Tarih","Olay","Fiyat"]

veri = veri.astype({"Fiyat": float}) ## silinecek pickleden okunduğu için konuldu

veri

In [ ]:
ilk_fiyat =veri.loc[0]["Fiyat"]
son_fiyat = veri.loc[len(veri.index)-1]["Fiyat"]
100*(son_fiyat-ilk_fiyat)/ilk_fiyat

In [ ]:
def hizli_simulasyon(fiyat):  
    if kunye['yukselis_aksiyon_fiyati'] > fiyat > kunye['inis_aksiyon_fiyati']:
        return

    Operation_Research(fiyat , False)
    Gelen_Degerleri_Kaydet(False)
    return

#Baslangic_Degerlerini_Ata(basilacak_mi=False)
#Gelen_Degerleri_Kaydet(False)

#veri.apply(lambda x: hizli_simulasyon(x["Fiyat"]) , axis=1)
#clear_output(wait=True)
#Mevcut_Durum()

In [ ]:
ncd_listesi = []
for i in range(1,10):
    Baslangic_Degerlerini_Ata(basilacak_mi=False)
    hiperparametreler["fuzyona_girecek_yuzde"] = i

    Gelen_Degerleri_Kaydet(False)
    a = veri.apply(lambda x: hizli_simulasyon(x["Fiyat"]) , axis=1)

    ncd = kunye["p1_miktari"]*(veri.iloc[-1]["Fiyat"])*hiperparametreler["komisyon_katsayisi"] + kunye["p2_miktari"]
    ncd_listesi.append([i ,ncd])
    clear_output(wait=True)
    sonuc = pd.DataFrame(ncd_listesi , columns=["i", "ncd"])
    print("Max NCD" ,sonuc["ncd"].max())
    sonuc

In [ ]:
def detayli_simulasyon(x):
    
    x["parca_sayisi"] = len(satis_plani.index) + len(alis_plani.index)

    fiyat = x["Fiyat"]    
    if kunye['yukselis_aksiyon_fiyati'] > fiyat > kunye['inis_aksiyon_fiyati']:
        x["son_aksiyon"] = "pas gecildi"
        x["ncd"] = kunye["p1_miktari"]*fiyat*hiperparametreler["komisyon_katsayisi"] + kunye["p2_miktari"]
        return x

    Operation_Research(fiyat , False)
    Gelen_Degerleri_Kaydet(False)

    ncd = kunye["nakde_cevrilebilir_deger"]
    clear_output()
    print(kunye["nakde_cevrilebilir_deger"])
    print(x['Tarih'])
    
    x["son_aksiyon"] = kunye["son_aksiyon"]
    x["ncd"] = kunye["nakde_cevrilebilir_deger"]
    return x

Baslangic_Degerlerini_Ata(basilacak_mi=False)
Gelen_Degerleri_Kaydet(False)

veri2 = veri.apply(lambda x: detayli_simulasyon(x) , axis=1)

In [ ]:
Mevcut_Durum()

In [ ]:
ax = veri2.plot('Tarih','ncd' ,figsize=(30,8) , color ='b')
ax1 = ax.twinx()
veri.plot('Tarih','Fiyat' ,figsize=(30,8),ax=ax1, color='r' , title='Fiyat - Nakde Çevrilebilir Değer Kıyaslaması')


ax2 = veri2.plot('Tarih','parca_sayisi' ,figsize=(30,8) , color ='b')
ax3 = ax2.twinx()
veri.plot('Tarih','Fiyat' ,figsize=(30,8),ax=ax3, color='r' , title='Fiyat - Toplam Parça Sayısı')

In [ ]:
veri2.to_pickle("drive/MyDrive/my_data.pkl")

In [ ]:
fig = ax.get_figure()
fig.savefig('/content/drive/MyDrive/figure.pdf')